
# Data Analysis with Python

## Project: Delivery Time Deviation Prediction in Logistics

This notebook performs Python-based data analysis for the logistics dataset.  
It includes:

1. Loading and checking the dataset.
2. Creating proxy variables to match the project requirements.
3. Pandas-based aggregation and correlation analysis.
4. Feature engineering with lag delivery and rolling average.
5. Time-based train-test split.
6. Regression modeling using five models:
   - Linear Regression
   - Ridge Regression
   - Decision Tree Regressor
   - Random Forest Regressor
   - XGBoost Regressor
7. Model evaluation using MAE, RMSE, and MAPE.

## Important Note on Dataset

The original project idea mentions `distance`, `vehicle`, `delivery duration`, `zone`, and `route`.  
However, the selected dataset does not contain these columns directly. Therefore, this notebook uses proxy variables:

| Original Concept | Proxy Variable Used |
|---|---|
| Distance | `distance_proxy` from `eta_variation_hours` |
| Vehicle | `vehicle_proxy` from driver behavior and fuel consumption |
| Delivery duration | `final_delivery_time_hours` from lead time and logistics time variables |
| Zone | `gps_zone` from GPS latitude and longitude |
| Route | `route_proxy` from GPS zone and route risk group |
| Late delivery | `is_late` from `delivery_time_deviation > 0` |



## 1. Import Libraries and Load Dataset


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)

cleaned_file = Path("cleaned_dynamic_supply_chain_logistics_dataset.csv")
original_file = Path("dynamic_supply_chain_logistics_dataset.csv")

if cleaned_file.exists():
    df = pd.read_csv(cleaned_file)
    print("Loaded cleaned dataset.")
elif original_file.exists():
    df = pd.read_csv(original_file)
    print("Loaded original dataset.")
else:
    raise FileNotFoundError("Dataset file not found. Please put the CSV file in the same folder as this notebook.")

print("Dataset shape:", df.shape)
display(df.head())



## 2. Basic Data Overview


In [ ]:

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicated rows:", df.duplicated().sum())

print("\nDescriptive statistics:")
display(df.describe(include="all"))



## 3. Prepare Dataset and Create Proxy Variables

This section creates the variables needed for Python analysis and modeling.


In [ ]:

# Convert timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# Sort data by time for time-based analysis
df = df.sort_values("timestamp").reset_index(drop=True)

# Target variable
target_col = "delivery_time_deviation"

# Time-based features
df["hour_of_day"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["day_name"] = df["timestamp"].dt.day_name()
df["month"] = df["timestamp"].dt.month
df["month_period"] = df["timestamp"].dt.to_period("M").astype(str)
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["is_peak_hour"] = df["hour_of_day"].isin([7, 8, 9, 17, 18, 19]).astype(int)

# Late delivery flag
df["is_late"] = (df[target_col] > 0).astype(int)

# Distance proxy
df["distance_proxy"] = df["eta_variation_hours"]

df["distance_group"] = pd.qcut(
    df["distance_proxy"],
    q=5,
    labels=["Very Short", "Short", "Medium", "Long", "Very Long"],
    duplicates="drop"
)

# Vehicle proxy
df["vehicle_score_proxy"] = (
    df["driver_behavior_score"] * 0.6 +
    (100 - df["fuel_consumption_rate"]) * 0.4
)

df["vehicle_proxy"] = pd.qcut(
    df["vehicle_score_proxy"],
    q=4,
    labels=["Low Performance", "Medium Performance", "Good Performance", "High Performance"],
    duplicates="drop"
)

# GPS-based zone
df["lat_bin"] = pd.cut(df["vehicle_gps_latitude"], bins=5, labels=False)
df["long_bin"] = pd.cut(df["vehicle_gps_longitude"], bins=5, labels=False)
df["gps_zone"] = df["lat_bin"].astype(str) + "_" + df["long_bin"].astype(str)

# Route risk group
df["route_risk_group"] = pd.qcut(
    df["route_risk_level"],
    q=5,
    labels=["Very Low Risk", "Low Risk", "Medium Risk", "High Risk", "Very High Risk"],
    duplicates="drop"
)

# Route proxy
df["route_proxy"] = df["gps_zone"].astype(str) + "_" + df["route_risk_group"].astype(str)

# Delivery duration proxy
df["final_delivery_time_hours"] = (
    df["lead_time_days"] * 24
    + df["loading_unloading_time"]
    + df["customs_clearance_time"]
    + df["eta_variation_hours"]
)

# Keep valid duration only
df = df[df["final_delivery_time_hours"] > 0].reset_index(drop=True)

display(df[[
    "timestamp",
    "hour_of_day",
    "distance_proxy",
    "distance_group",
    "vehicle_proxy",
    "gps_zone",
    "route_proxy",
    "final_delivery_time_hours",
    "delivery_time_deviation",
    "is_late"
]].head())



## 4. Pandas Analysis: Average Delivery Time by Distance, Hour, and Vehicle

This answers the requirement:

> Calculate average delivery time by distance, hour, and vehicle.

Because the dataset does not contain direct distance and vehicle type, this analysis uses:
- `distance_group` as distance proxy.
- `hour_of_day` from timestamp.
- `vehicle_proxy` from driver behavior and fuel consumption.
- `final_delivery_time_hours` as delivery duration proxy.


In [ ]:

avg_delivery_analysis = df.groupby(
    ["distance_group", "hour_of_day", "vehicle_proxy"]
).agg(
    total_records=("final_delivery_time_hours", "count"),
    avg_delivery_time_hours=("final_delivery_time_hours", "mean"),
    avg_delivery_deviation=("delivery_time_deviation", "mean")
).reset_index()

avg_delivery_analysis = avg_delivery_analysis.sort_values(
    by="avg_delivery_time_hours",
    ascending=False
)

display(avg_delivery_analysis.head(20))



## 5. Pandas Analysis: Late Delivery Rate by Zone

This answers the requirement:

> Analyze late delivery rate by zone.

Since the dataset does not contain a direct delivery zone column, `gps_zone` is used.


In [ ]:

late_rate_by_zone = df.groupby("gps_zone").agg(
    total_records=("is_late", "count"),
    late_records=("is_late", "sum"),
    late_delivery_rate=("is_late", "mean"),
    avg_delivery_deviation=("delivery_time_deviation", "mean")
).reset_index()

late_rate_by_zone["late_delivery_rate_percent"] = late_rate_by_zone["late_delivery_rate"] * 100

late_rate_by_zone = late_rate_by_zone.sort_values(
    by="late_delivery_rate_percent",
    ascending=False
)

display(late_rate_by_zone.head(20))



## 6. Pandas Analysis: Correlation Between Distance and Duration

This answers the requirement:

> Query correlation between distance and duration.

Because the dataset does not contain direct distance and duration:
- `distance_proxy` is used for distance.
- `final_delivery_time_hours` is used for duration.


In [ ]:

corr_distance_duration = df["distance_proxy"].corr(df["final_delivery_time_hours"])
corr_distance_deviation = df["distance_proxy"].corr(df["delivery_time_deviation"])

print("Correlation between distance_proxy and final_delivery_time_hours:", corr_distance_duration)
print("Correlation between distance_proxy and delivery_time_deviation:", corr_distance_deviation)



## 7. Pandas Analysis: Aggregate by Popular Routes

This answers the requirement:

> Aggregate by popular routes.

Since the dataset does not contain route ID, `route_proxy` is created from GPS zone and route risk group.


In [ ]:

popular_routes = df.groupby("route_proxy").agg(
    total_records=("route_proxy", "count"),
    avg_delivery_time_hours=("final_delivery_time_hours", "mean"),
    avg_delivery_deviation=("delivery_time_deviation", "mean"),
    avg_delay_probability=("delay_probability", "mean"),
    avg_traffic_congestion=("traffic_congestion_level", "mean")
).reset_index()

popular_routes = popular_routes.sort_values(by="total_records", ascending=False)

display(popular_routes.head(10))



## 8. Additional Python Analysis: Target Variable Summary

This section analyzes the main target variable: `delivery_time_deviation`.


In [ ]:

print("Target variable summary:")
display(df[target_col].describe())

plt.figure(figsize=(8, 5))
sns.histplot(df[target_col], bins=40, kde=True)
plt.title("Distribution of Delivery Time Deviation")
plt.xlabel("Delivery Time Deviation")
plt.ylabel("Frequency")
plt.show()



## 9. Additional Python Analysis: Average Deviation by Risk Classification


In [ ]:

risk_summary = df.groupby("risk_classification").agg(
    total_records=("delivery_time_deviation", "count"),
    avg_delivery_deviation=("delivery_time_deviation", "mean"),
    median_delivery_deviation=("delivery_time_deviation", "median"),
    min_delivery_deviation=("delivery_time_deviation", "min"),
    max_delivery_deviation=("delivery_time_deviation", "max")
).reset_index()

risk_summary = risk_summary.sort_values(by="avg_delivery_deviation", ascending=False)

display(risk_summary)

plt.figure(figsize=(8, 5))
sns.barplot(
    data=risk_summary,
    x="risk_classification",
    y="avg_delivery_deviation"
)
plt.title("Average Delivery Time Deviation by Risk Classification")
plt.xlabel("Risk Classification")
plt.ylabel("Average Delivery Time Deviation")
plt.show()



## 10. Feature Engineering for Machine Learning

This section creates features used for regression modeling.

Required feature engineering:
- Lag delivery features.
- Rolling average delivery features.
- Interaction features.


In [ ]:

# Lag delivery features
df["lag_delivery_deviation_1"] = df[target_col].shift(1)
df["lag_delivery_deviation_3"] = df[target_col].shift(3)
df["lag_delivery_deviation_7"] = df[target_col].shift(7)

# Rolling average features
# shift(1) is used to avoid data leakage
df["rolling_avg_deviation_3"] = df[target_col].shift(1).rolling(window=3).mean()
df["rolling_avg_deviation_7"] = df[target_col].shift(1).rolling(window=7).mean()
df["rolling_avg_deviation_14"] = df[target_col].shift(1).rolling(window=14).mean()

# Interaction features
df["traffic_intensity_proxy"] = df["traffic_congestion_level"] * df["is_peak_hour"]

df["route_complexity"] = (
    df["traffic_congestion_level"]
    + df["route_risk_level"]
    + df["weather_condition_severity"] * 10
) / 3

df["traffic_x_weather"] = (
    df["traffic_congestion_level"] *
    df["weather_condition_severity"]
)

df["traffic_x_route_risk"] = (
    df["traffic_congestion_level"] *
    df["route_risk_level"]
)

df["eta_variation_x_peak_hour"] = (
    df["eta_variation_hours"] *
    df["is_peak_hour"]
)

# Drop missing rows created by lag and rolling features
df_model = df.dropna().reset_index(drop=True)

print("Dataset shape after feature engineering:", df_model.shape)

display(df_model[[
    target_col,
    "lag_delivery_deviation_1",
    "rolling_avg_deviation_3",
    "traffic_intensity_proxy",
    "route_complexity"
]].head())



## 11. Prepare Features and Target for Modeling


In [ ]:

feature_cols = [
    "vehicle_gps_latitude",
    "vehicle_gps_longitude",
    "fuel_consumption_rate",
    "eta_variation_hours",
    "traffic_congestion_level",
    "warehouse_inventory_level",
    "loading_unloading_time",
    "handling_equipment_availability",
    "order_fulfillment_status",
    "weather_condition_severity",
    "port_congestion_level",
    "shipping_costs",
    "supplier_reliability_score",
    "lead_time_days",
    "historical_demand",
    "iot_temperature",
    "cargo_condition_status",
    "route_risk_level",
    "customs_clearance_time",
    "driver_behavior_score",
    "fatigue_monitoring_score",
    "disruption_likelihood_score",
    "delay_probability",
    "hour_of_day",
    "day_of_week",
    "month",
    "is_weekend",
    "is_peak_hour",
    "distance_proxy",
    "vehicle_score_proxy",
    "final_delivery_time_hours",
    "lag_delivery_deviation_1",
    "lag_delivery_deviation_3",
    "lag_delivery_deviation_7",
    "rolling_avg_deviation_3",
    "rolling_avg_deviation_7",
    "rolling_avg_deviation_14",
    "traffic_intensity_proxy",
    "route_complexity",
    "traffic_x_weather",
    "traffic_x_route_risk",
    "eta_variation_x_peak_hour"
]

X = df_model[feature_cols]
y = df_model[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)



## 12. Time-Based Train-Test Split

The dataset is sorted by timestamp.  
The first 80% of records are used for training, and the last 20% are used for testing.

This is more realistic than random split because the model should predict future records based on past data.


In [ ]:

split_index = int(len(df_model) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)



## 13. Define Evaluation Metrics

Metrics:
- MAE: Mean Absolute Error.
- RMSE: Root Mean Squared Error.
- MAPE: Mean Absolute Percentage Error.

MAPE is useful because it gives percentage-based error.  
However, because `delivery_time_deviation` can be negative or close to zero, this notebook uses `abs(y_true) + epsilon` in the denominator.


In [ ]:

from sklearn.metrics import mean_absolute_error, mean_squared_error

def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    epsilon = 1e-8
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + epsilon))) * 100

    return mae, rmse, mape



## 14. Train Five Regression Models


In [ ]:

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(random_state=42, max_depth=8),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

# Add XGBoost if installed
try:
    from xgboost import XGBRegressor

    models["XGBoost"] = XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        random_state=42,
        objective="reg:squarederror"
    )
except ImportError:
    print("XGBoost is not installed. Run: pip install xgboost")


In [ ]:

model_results = []
predictions = {}

for model_name, model in models.items():
    print("Training:", model_name)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae, rmse, mape = calculate_metrics(y_test, y_pred)

    model_results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape
    })

    predictions[model_name] = y_pred

results_df = pd.DataFrame(model_results).sort_values(by="MAPE (%)")

display(results_df)



## 15. Model Comparison Visualization


In [ ]:

plt.figure(figsize=(9, 5))

sns.barplot(
    data=results_df,
    x="Model",
    y="MAPE (%)"
)

plt.title("Model Comparison Based on MAPE")
plt.xlabel("Model")
plt.ylabel("MAPE (%)")
plt.xticks(rotation=30)
plt.show()



## 16. Actual vs Predicted Comparison

This section compares actual and predicted values for the best model.


In [ ]:

best_model_name = results_df.iloc[0]["Model"]
best_pred = predictions[best_model_name]

print("Best model:", best_model_name)

comparison_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": best_pred,
    "Error": y_test.values - best_pred
})

display(comparison_df.head(20))

plt.figure(figsize=(8, 6))

plt.scatter(y_test, best_pred, alpha=0.4)
plt.xlabel("Actual Delivery Time Deviation")
plt.ylabel("Predicted Delivery Time Deviation")
plt.title(f"Actual vs Predicted - {best_model_name}")
plt.grid(True, alpha=0.3)
plt.show()



## 17. Feature Importance

Feature importance is available for tree-based models such as Decision Tree, Random Forest, and XGBoost.


In [ ]:

best_model = models[best_model_name]

if hasattr(best_model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "Feature": feature_cols,
        "Importance": best_model.feature_importances_
    }).sort_values(by="Importance", ascending=False)

    display(importance_df.head(15))

    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=importance_df.head(15),
        x="Importance",
        y="Feature"
    )

    plt.title(f"Top 15 Feature Importance - {best_model_name}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.show()
else:
    print(f"{best_model_name} does not provide feature_importances_.")



## 18. Export Analysis Results


In [ ]:

# Export model comparison results
results_df.to_csv("python_model_comparison_results.csv", index=False)

# Export predictions of best model
prediction_output = df_model.iloc[split_index:].copy()
prediction_output["best_model_prediction"] = best_pred
prediction_output["prediction_error"] = prediction_output[target_col] - prediction_output["best_model_prediction"]

prediction_output.to_csv("python_prediction_results.csv", index=False)

print("Exported files:")
print("- python_model_comparison_results.csv")
print("- python_prediction_results.csv")



## 19. Summary

This notebook completed Python-based analysis for the logistics delivery dataset.

Main completed tasks:

1. Loaded and checked the dataset.
2. Created proxy variables for distance, vehicle, zone, route, and delivery duration.
3. Analyzed average delivery time by distance group, hour, and vehicle proxy.
4. Analyzed late delivery rate by GPS-based zone.
5. Calculated correlation between distance proxy and delivery duration proxy.
6. Aggregated popular route proxies.
7. Created lag delivery and rolling average features.
8. Applied time-based train-test split.
9. Trained five regression models.
10. Evaluated models using MAE, RMSE, and MAPE.
11. Exported prediction and model comparison results.

The results from this notebook can be used in the final report and presentation.
